This notebook computes differences between the drifters velocities and the velocities from the original L3-SWOT product and the DMD L3-SWOT product.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import xarray as xr
import xesmf as xe

In [3]:
drifter003_ds = xr.open_zarr("data/drifters003_drogue15m_swath_CalVal.zarr")
drifter016_ds = xr.open_zarr("data/drifters016_drogue15m_swath_CalVal.zarr")

l3_swot_003_ds = xr.open_zarr("data/SWOT_L3_Expert_v3_0_pass003_cg.zarr")
l3_swot_016_ds = xr.open_zarr("data/SWOT_L3_Expert_v3_0_pass016_cg.zarr")

In [4]:
l3_swot_003_ds

<xarray.Dataset> Size: 1GB
Dimensions:               (cycle_num: 98, num_lines: 390, num_pixels: 69)
Coordinates:
  * cycle_num             (cycle_num) int64 784B 474 475 476 478 ... 576 577 578
  * num_lines             (num_lines) int64 3kB 6988 6989 6990 ... 7376 7377
    time                  (cycle_num, num_lines) datetime64[ns] 306kB dask.array<chunksize=(10, 390), meta=np.ndarray>
  * num_pixels            (num_pixels) int64 552B 0 1 2 3 4 5 ... 64 65 66 67 68
    latitude              (num_lines, num_pixels) float64 215kB dask.array<chunksize=(390, 69), meta=np.ndarray>
    longitude             (num_lines, num_pixels) float64 215kB dask.array<chunksize=(390, 69), meta=np.ndarray>
Data variables: (12/51)
    calibration           (cycle_num, num_lines, num_pixels) float64 21MB dask.array<chunksize=(10, 390, 69), meta=np.ndarray>
    cross_track_distance  (num_pixels) float64 552B dask.array<chunksize=(69,), meta=np.ndarray>
    dac                   (cycle_num, num_lines, num_pixels) float64 21MB dask.array<chunksize=(10, 390, 69), meta=np.ndarray>
    eke_cg_fp             (cycle_num, num_lines, num_pixels) float64 21MB dask.array<chunksize=(10, 390, 69), meta=np.ndarray>
    eke_cg_gw             (cycle_num, num_lines, num_pixels) float64 21MB dask.array<chunksize=(10, 390, 69), meta=np.ndarray>
    eke_cg_mb             (cycle_num, num_lines, num_pixels) float64 21MB dask.array<chunksize=(10, 390, 69), meta=np.ndarray>
    ...                    ...
    vcga_fp               (cycle_num, num_lines, num_pixels) float64 21MB dask.array<chunksize=(10, 390, 69), meta=np.ndarray>
    vcga_gw               (cycle_num, num_lines, num_pixels) float64 21MB dask.array<chunksize=(10, 390, 69), meta=np.ndarray>
    vcga_mb               (cycle_num, num_lines, num_pixels) float64 21MB dask.array<chunksize=(10, 390, 69), meta=np.ndarray>
    vgos_filtered         (cycle_num, num_lines, num_pixels) float64 21MB dask.array<chunksize=(10, 390, 69), meta=np.ndarray>
    vgosa_filtered        (cycle_num, num_lines, num_pixels) float64 21MB dask.array<chunksize=(10, 390, 69), meta=np.ndarray>
    vgosa_unfiltered      (cycle_num, num_lines, num_pixels) float64 21MB dask.array<chunksize=(10, 390, 69), meta=np.ndarray>
Attributes: (12/43)
    Conventions:                     CF-1.9
    Metadata_Conventions:            Unidata Dataset Discovery v1.0
    cdm_data_type:                   Swath
    comment:                         Sea Surface Height measured by Altimetry
    data_used:                       SWOT KaRIn L2_LR_SSH PGC0/PIC0/PIC2/PID0...
    doi:                             https://doi.org/10.24400/527896/A01-2023...
    ...                              ...
    geospatial_lon_max:              359.999928
    date_modified:                   2025-11-24T16:24:38Z
    history:                         2025-11-24T16:24:38Z: Created by DUACS K...
    date_created:                    2025-11-24T16:24:38Z
    date_issued:                     2025-11-24T16:24:38Z
    temporality:                     reproc

In [6]:
time_3d = (
    l3_swot_003_ds["time"]
    .astype("datetime64[s]")
    .astype(int)
    .expand_dims(dim="num_pixels", axis=-1)
    .isel(num_pixels=[0] * l3_swot_003_ds.sizes["num_pixels"])
    .assign_coords(num_pixels=l3_swot_003_ds.num_pixels)
    .chunk(dict(((d, c[0]) for d, c in zip(l3_swot_003_ds["ugos_filtered"].dims, l3_swot_003_ds["ugos_filtered"].chunks))))
)

In [7]:
l3_swot_003_ds["time"]

<xarray.DataArray 'time' (cycle_num: 98, num_lines: 390)> Size: 306kB
dask.array<open_dataset-time, shape=(98, 390), dtype=datetime64[ns], chunksize=(10, 390), chunktype=numpy.ndarray>
Coordinates:
  * cycle_num  (cycle_num) int64 784B 474 475 476 478 479 ... 575 576 577 578
  * num_lines  (num_lines) int64 3kB 6988 6989 6990 6991 ... 7374 7375 7376 7377
    time       (cycle_num, num_lines) datetime64[ns] 306kB dask.array<chunksize=(10, 390), meta=np.ndarray>
Attributes:
    comment:             Time of measurement in seconds in the UTC time scale...
    leap_second:         0000-00-00T00:00:00Z
    long_name:           time in UTC
    standard_name:       time
    tai_utc_difference:  37.0

In [9]:
def interp_grid_vel_to_points(drifter_ds, pass_ds, u_label, v_label):
    # 1. Interpolate in space from satellite swath grid to drifter positions, also interpolates overpass time for step 2

    pass_ds = pass_ds.reset_coords("time")

    time_3d = (
        pass_ds["time"]
        .astype("datetime64[s]")
        .astype(int)
        .expand_dims(dim="num_pixels", axis=-1)
        .isel(num_pixels=[0] * pass_ds.sizes["num_pixels"])
        .assign_coords(num_pixels=pass_ds.num_pixels)
        .chunk(dict(((d, c[0]) for d, c in zip(pass_ds[u_label].dims, pass_ds[u_label].chunks))))
    )

    ds_src = xr.Dataset(
        {
            "u": pass_ds[u_label],
            "v": pass_ds[v_label],
            "time": time_3d
        },
        coords={
            "lat": (("num_lines", "num_pixels"), pass_ds.latitude.values),
            "lon": (("num_lines", "num_pixels"), pass_ds.longitude.values),
        },
    )

    ds_tgt = xr.Dataset(
        coords={
            "lat": ("points", drifter_ds.latitude.values),
            "lon": ("points", drifter_ds.longitude.values),
        }
    )

    regridder = xe.Regridder(ds_src, ds_tgt, method="bilinear", locstream_out=True)

    u_space = regridder(pass_ds[u_label])
    v_space = regridder(pass_ds[v_label])
    t_space = regridder(time_3d)

    # 2. Interpolate in time from satellite overpass times to drifter times

    u_interp = xr.apply_ufunc(
        np.interp,
        drifter_ds.time.astype("datetime64[s]").astype(int),  # x: target drifter times, shape (points,)
        t_space.chunk(dict(cycle_num=-1)),  # xp: satellite time per cycle and point, shape (cycle_num, points)
        u_space.chunk(dict(cycle_num=-1)),  # fp: velocities per cycle and point, shape (cycle_num, points)
        input_core_dims=[[], ["cycle_num"], ["cycle_num"]],
        vectorize=True,
        dask="parallelized"
    )
    v_interp = xr.apply_ufunc(
        np.interp,
        drifter_ds.time.astype("datetime64[s]").astype(int),
        t_space.chunk(dict(cycle_num=-1)),
        v_space.chunk(dict(cycle_num=-1)),
        input_core_dims=[[], ["cycle_num"], ["cycle_num"]],
        vectorize=True,
        dask="parallelized"
    )

    u_interp, v_interp = u_interp.values, v_interp.values

    return u_interp, v_interp

In [10]:
l3_003_ug_interp, l3_003_vg_interp = interp_grid_vel_to_points(
    drifter003_ds, l3_swot_003_ds, "ugos_filtered", "vgos_filtered"
)
l3_016_ug_interp, l3_016_vg_interp = interp_grid_vel_to_points(
    drifter016_ds, l3_swot_016_ds, "ugos_filtered", "vgos_filtered"
)

l3_003_ucg_mb_interp, l3_003_vcg_mb_interp = interp_grid_vel_to_points(
    drifter003_ds, l3_swot_003_ds, "ucg_mb", "vcg_mb"
)
l3_016_ucg_mb_interp, l3_016_vcg_mb_interp = interp_grid_vel_to_points(
    drifter016_ds, l3_swot_016_ds, "ucg_mb", "vcg_mb"
)

l3_003_ucg_fp_interp, l3_003_vcg_fp_interp = interp_grid_vel_to_points(
    drifter003_ds, l3_swot_003_ds, "ucg_fp", "vcg_fp"
)
l3_016_ucg_fp_interp, l3_016_vcg_fp_interp = interp_grid_vel_to_points(
    drifter016_ds, l3_swot_016_ds, "ucg_fp", "vcg_fp"
)

l3_003_ucg_gw_interp, l3_003_vcg_gw_interp = interp_grid_vel_to_points(
    drifter003_ds, l3_swot_003_ds, "ucg_gw", "vcg_gw"
)
l3_016_ucg_gw_interp, l3_016_vcg_gw_interp = interp_grid_vel_to_points(
    drifter016_ds, l3_swot_016_ds, "ucg_gw", "vcg_gw"
)

In [11]:
drifter003_ds["ugos"] = ("points", l3_003_ug_interp)
drifter003_ds["vgos"] = ("points", l3_003_vg_interp)

drifter016_ds["ugos"] = ("points", l3_016_ug_interp)
drifter016_ds["vgos"] = ("points", l3_016_vg_interp)

drifter003_ds["ucg_mb"] = ("points", l3_003_ucg_mb_interp)
drifter003_ds["vcg_mb"] = ("points", l3_003_vcg_mb_interp)

drifter016_ds["ucg_mb"] = ("points", l3_016_ucg_mb_interp)
drifter016_ds["vcg_mb"] = ("points", l3_016_vcg_mb_interp)

drifter003_ds["ucg_fp"] = ("points", l3_003_ucg_fp_interp)
drifter003_ds["vcg_fp"] = ("points", l3_003_vcg_fp_interp)

drifter016_ds["ucg_fp"] = ("points", l3_016_ucg_fp_interp)
drifter016_ds["vcg_fp"] = ("points", l3_016_vcg_fp_interp)

drifter003_ds["ucg_gw"] = ("points", l3_003_ucg_gw_interp)
drifter003_ds["vcg_gw"] = ("points", l3_003_vcg_gw_interp)

drifter016_ds["ucg_gw"] = ("points", l3_016_ucg_gw_interp)
drifter016_ds["vcg_gw"] = ("points", l3_016_vcg_gw_interp)

In [12]:
drifter003_ds["ugos_diff"] = drifter003_ds["velocity_east"] - drifter003_ds["ugos"]
drifter003_ds["vgos_diff"] = drifter003_ds["velocity_north"] - drifter003_ds["vgos"]

drifter016_ds["ugos_diff"] = drifter016_ds["velocity_east"] - drifter016_ds["ugos"]
drifter016_ds["vgos_diff"] = drifter016_ds["velocity_north"] - drifter016_ds["vgos"]

drifter003_ds["ucg_mb_diff"] = drifter003_ds["velocity_east"] - drifter003_ds["ucg_mb"]
drifter003_ds["vcg_mb_diff"] = drifter003_ds["velocity_north"] - drifter003_ds["vcg_mb"]

drifter016_ds["ucg_mb_diff"] = drifter016_ds["velocity_east"] - drifter016_ds["ucg_mb"]
drifter016_ds["vcg_mb_diff"] = drifter016_ds["velocity_north"] - drifter016_ds["vcg_mb"]

drifter003_ds["ucg_fp_diff"] = drifter003_ds["velocity_east"] - drifter003_ds["ucg_fp"]
drifter003_ds["vcg_fp_diff"] = drifter003_ds["velocity_north"] - drifter003_ds["vcg_fp"]

drifter016_ds["ucg_fp_diff"] = drifter016_ds["velocity_east"] - drifter016_ds["ucg_fp"]
drifter016_ds["vcg_fp_diff"] = drifter016_ds["velocity_north"] - drifter016_ds["vcg_fp"]

drifter003_ds["ucg_gw_diff"] = drifter003_ds["velocity_east"] - drifter003_ds["ucg_gw"]
drifter003_ds["vcg_gw_diff"] = drifter003_ds["velocity_north"] - drifter003_ds["vcg_gw"]

drifter016_ds["ucg_gw_diff"] = drifter016_ds["velocity_east"] - drifter016_ds["ucg_gw"]
drifter016_ds["vcg_gw_diff"] = drifter016_ds["velocity_north"] - drifter016_ds["vcg_gw"]

In [13]:
drifter003_ds["gos_diff"] = (
    drifter003_ds["ugos_diff"] ** 2 + drifter003_ds["vgos_diff"] ** 2
) ** 0.5
drifter016_ds["gos_diff"] = (
    drifter016_ds["ugos_diff"] ** 2 + drifter016_ds["vgos_diff"] ** 2
) ** 0.5

drifter003_ds["cg_mb_diff"] = (
    drifter003_ds["ucg_mb_diff"] ** 2 + drifter003_ds["vcg_mb_diff"] ** 2
) ** 0.5
drifter016_ds["cg_mb_diff"] = (
    drifter016_ds["ucg_mb_diff"] ** 2 + drifter016_ds["vcg_mb_diff"] ** 2
) ** 0.5

drifter003_ds["cg_fp_diff"] = (
    drifter003_ds["ucg_fp_diff"] ** 2 + drifter003_ds["vcg_fp_diff"] ** 2
) ** 0.5
drifter016_ds["cg_fp_diff"] = (
    drifter016_ds["ucg_fp_diff"] ** 2 + drifter016_ds["vcg_fp_diff"] ** 2
) ** 0.5

drifter003_ds["cg_gw_diff"] = (
    drifter003_ds["ucg_gw_diff"] ** 2 + drifter003_ds["vcg_gw_diff"] ** 2
) ** 0.5
drifter016_ds["cg_gw_diff"] = (
    drifter016_ds["ucg_gw_diff"] ** 2 + drifter016_ds["vcg_gw_diff"] ** 2
) ** 0.5

In [14]:
drifter_ds = xr.concat([drifter003_ds, drifter016_ds], dim="points")

In [15]:
drifter_ds = drifter_ds.drop_vars(["drifter_id", "drifter_type"])

In [16]:
drifter_ds.chunk(points=-1).to_zarr("data/drifters_drogue15m_swath_CalVal_diff.zarr", mode="w")

/Users/bertrava/miniforge3/envs/swot-cyclogeo/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
